# 30 — Helicorder browser GUI (Z channels)

Dropdown station + day step buttons; uses daily miniSEED written by notebook 20.

In [ ]:
%run 00_config.ipynb
from obspy import read
import ipywidgets as widgets
from IPython.display import display, clear_output

In [ ]:
wf_dir = os.path.join(ROOT, "waveforms_daily")
assert os.path.isdir(wf_dir), "Run 20_waveforms_daily.ipynb first"

files = sorted([f for f in os.listdir(wf_dir) if f.endswith(".Z.mseed")])
assert files, "No daily files found. Run 20_waveforms_daily.ipynb."

stations = sorted(set(f.split(".")[1] for f in files))
days = sorted(set(f.split(".")[2] for f in files))  # YYYY-MM-DD strings

station_dd = widgets.Dropdown(options=stations, description="Station:")
day_dd = widgets.Dropdown(options=days, description="Day:")
btn_prev = widgets.Button(description="← Prev day")
btn_next = widgets.Button(description="Next day →")
out = widgets.Output()

def plot_day(station, day_str):
    matches = [f for f in os.listdir(wf_dir) if f.endswith(f".{station}.{day_str}.Z.mseed")]
    if not matches:
        print("No file for", station, day_str)
        return

    st = Stream()
    for m in matches:
        st += read(os.path.join(wf_dir, m))
    st.sort(keys=["network","station","location","channel","starttime"])

    for tr in st:
        tr.plot(type="dayplot", interval=60, title=f"{tr.id} {day_str}")

def redraw(*args):
    with out:
        clear_output(wait=True)
        plot_day(station_dd.value, day_dd.value)

def step_day(delta):
    opts = list(day_dd.options)
    i = opts.index(day_dd.value)
    j = max(0, min(len(opts)-1, i+delta))
    day_dd.value = opts[j]

btn_prev.on_click(lambda b: step_day(-1))
btn_next.on_click(lambda b: step_day(+1))
station_dd.observe(redraw, names="value")
day_dd.observe(redraw, names="value")

display(widgets.HBox([station_dd, btn_prev, btn_next, day_dd]))
display(out)
redraw()
